# `bisect` & Sorting (Timsort)

Python combines two rock-solid tools for ordered sequences:
1. **`bisect`**: $O(\log n)$ binary search on already-sorted sequences (supports `key=` since Python 3.10!).
2. **Timsort** (`list.sort()` and `sorted()`): hybrid merge + insertion sort with $O(n \log n)$ worst-case, **$O(n)$ best-case on nearly sorted data**, and **guaranteed stability**.

In [1]:
import bisect
from operator import itemgetter

---
## 1. `bisect_left` vs. `bisect_right`

Given a sorted sequence `a` and target `x`:
- **`bisect_left(a, x)`**: leftmost index where `x` could be inserted while preserving order (all elements before it are `< x`).
- **`bisect_right(a, x)`** (alias `bisect`): rightmost index where `x` could be inserted (all elements before it are `<= x`).

> **Complexity note:** Searching (`bisect_left` / `bisect_right`) is **$O(\log n)$**, whereas `insort_left` / `insort_right` is **$O(n)$** because inserting into a Python `list` shifts subsequent elements!

In [2]:
arr = [10, 20, 20, 20, 30, 40]

left = bisect.bisect_left(arr, 20)
right = bisect.bisect_right(arr, 20)
print(f"20 spans indices [{left}, {right}) -> count = {right - left}")

# Exact membership check in O(log n)
def binary_search(sorted_seq, target) -> int:
    i = bisect.bisect_left(sorted_seq, target)
    if i != len(sorted_seq) and sorted_seq[i] == target:
        return i
    return -1

print("binary_search(30):", binary_search(arr, 30), "| binary_search(25):", binary_search(arr, 25))

# Classic use case: numeric bucket / threshold lookup
def score_to_tier(score: float, breakpoints=(0.5, 0.75, 0.9), tiers=("Low", "Medium", "High", "Top")) -> str:
    idx = bisect.bisect_right(breakpoints, score)
    return tiers[idx]

for s in [0.2, 0.5, 0.82, 0.95]:
    print(f"score={s:.2f} -> {score_to_tier(s)}")

# Python 3.10+ key= parameter on sorted list of dicts
events = [{"ts": 100, "msg": "start"}, {"ts": 250, "msg": "train"}, {"ts": 400, "msg": "eval"}]
idx = bisect.bisect_left(events, 250, key=lambda e: e["ts"])
print("First event with ts >= 250:", events[idx])

20 spans indices [1, 4) -> count = 3
binary_search(30): 4 | binary_search(25): -1
score=0.20 -> Low
score=0.50 -> Medium
score=0.82 -> High
score=0.95 -> Top
First event with ts >= 250: {'ts': 250, 'msg': 'train'}


---
## 2. Sorting & Timsort Stability

- **`list.sort(key=..., reverse=...)`**: sorts **in-place**, returns `None`, $O(1)$ extra space (besides Timsort's temp buffer).
- **`sorted(iterable, key=..., reverse=...)`**: works on **any** iterable, returns a new `list`.
- **Stability guarantee:** Elements with equal keys preserve their original relative order! This enables clean multi-pass mixed-direction sorting (e.g., sort by score **descending**, then name **ascending**).
- **`operator.itemgetter` / `attrgetter`** are implemented in C and noticeably faster than `lambda` for large lists.

In [3]:
candidates = [
    {"name": "Charlie", "score": 92, "yoe": 4},
    {"name": "Alice", "score": 95, "yoe": 6},
    {"name": "Bob", "score": 92, "yoe": 5},
    {"name": "Diana", "score": 95, "yoe": 3},
]

# Single-pass tuple key: score DESC, then name ASC
ranked = sorted(candidates, key=lambda c: (-c["score"], c["name"]))
print("Score DESC, Name ASC:", [(c["name"], c["score"]) for c in ranked])

# Multi-pass leveraging Timsort stability (useful when secondary key isn't numeric!)
# Goal: score DESC, yoe DESC, name ASC -> sort from least-significant key to most-significant!
multi = list(candidates)
multi.sort(key=itemgetter("name"))                      # 3rd key: name ASC
multi.sort(key=itemgetter("yoe"), reverse=True)         # 2nd key: yoe DESC
multi.sort(key=itemgetter("score"), reverse=True)       # 1st key: score DESC
print("Multi-pass stable sort:", [(c["name"], c["score"], c["yoe"]) for c in multi])

Score DESC, Name ASC: [('Alice', 95), ('Diana', 95), ('Bob', 92), ('Charlie', 92)]
Multi-pass stable sort: [('Alice', 95, 6), ('Diana', 95, 3), ('Bob', 92, 5), ('Charlie', 92, 4)]
